# Can an LLM Fix the Original Cross-Source Problem? / 大模型能不能解决最初的跨来源问题？

这个项目的起点：在 ISOT 上训练的模型达到 93–99%，但换到 WELFake（真新闻来自路透社以外的网站）只有 52–59%，因为它们学到的是 **"路透社的写作风格"**（`04`、`05`）。纽约时报的真新闻大多被判为假。

第 5 步的大模型只在 LIAR（一句话的陈述）上测试过。这里回到 **最初的问题**：让 Haiku 4.5 和 Sonnet 5.5 *不经过任何训练*，直接判断 WELFake test 的文章。它们没有见过 ISOT，所以没有机会学到"认路透社"这个捷径。

| 设置 | 选择 |
|---|---|
| 文章 | WELFake test 的 1,501 篇，按"假新闻 / 纽约时报 / Breitbart / 其他真新闻"分层抽样（种子 42） |
| 输入 | 标题 + 正文前 300 个词（正文中位数约 620 个词） |
| 提示词 | 中性："真新闻 vs 假新闻（捏造或虚假的报道）"，不提文风 |
| 比较 | 在 ISOT 上训练的模型（你的论文的 LSTM A0、调好的 BiLSTM A6、TF-IDF、DistilBERT）在 **同样的 1,501 篇** 上的结果 |

没有需要调的东西，所以只用 test。代码：`src/llm_crosssource.py`。

**需要注意**：WELFake 的标签也是按网站打的（某些网站的文章全算真，某些全算假），所以大模型的分数也可能部分来自"文风像不像假新闻网站"，而不只是核实事实。第 5 节专门检查这一点。

The project started from ISOT models scoring 93–99% in-domain but 52–59% on WELFake because they learned Reuters' style. Here Haiku and Sonnet judge 1,501 stratified WELFake test articles zero-shot (title + first 300 words), compared with the ISOT-trained models on the same articles.

## 1. Setup / 环境

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

sys.path.insert(0, str(Path.cwd().parent))

from src.analysis import paired_bootstrap_auc_diff
from src.baselines import bootstrap_ci, evaluate
from src.experiments import load_probs
from src.llm_crosssource import sample_test

LLM = Path('../results/llm')
pd.set_option('display.width', 220, 'display.max_colwidth', 120)

sample = sample_test().set_index('doc_id')
ids = sample.index
y = sample['label']
group = sample['group']
print(len(sample), 'articles'); print(group.value_counts().to_string())

def llm(model):
    d = np.load(LLM / f'llm_welfake_{model}.npz')
    return pd.Series(d['prob'], index=d['doc_id']).loc[ids]

def isot(model, config, seed):
    doc_ids, prob = load_probs(model, config, seed)['WELFake test']
    return pd.Series(prob, index=np.asarray(doc_ids).astype(str)).loc[ids]

P = {'LSTM, my paper (A0, 3 seeds mean)': pd.concat([isot('lstm', 'A0', s) for s in (42, 43, 44)], axis=1).mean(axis=1),
     'BiLSTM tuned (A6, 3 seeds mean)': pd.concat([isot('lstm', 'A6', s) for s in (42, 43, 44)], axis=1).mean(axis=1),
     'TF-IDF (ISOT)': isot('tfidf', 'C64', 0),
     'DistilBERT (ISOT, 3 seeds mean)': pd.concat([isot('distilbert', 'DistilBERT', s) for s in (42, 43, 44)], axis=1).mean(axis=1),
     'Haiku 4.5, zero-shot': llm('haiku'),
     'Sonnet 5.5, zero-shot': llm('sonnet')}
for m in ('haiku', 'sonnet'):
    rec = json.loads((LLM / f'llm_welfake_{m}.json').read_text())
    print(m, '| unparsed replies:', rec['unparsed'], '| tokens:', rec['usage'])

1501 articles
group
FAKE articles (all)         659
REAL: The New York Times    400
REAL: other outlets         301
REAL: Breitbart             141
haiku | unparsed replies: 0 | tokens: {'input_tokens': 879141, 'output_tokens': 13509}
sonnet | unparsed replies: 0 | tokens: {'input_tokens': 1247024, 'output_tokens': 13519}


ISOT 模型的 3 个种子取平均概率（和 `02`、`03` 中按种子分别报告的数字略有不同，但结论一样）。

The ISOT models' three seeds are averaged here.

## 2. Results on the same 1,501 articles / 同一批 1,501 篇文章上的结果

In [2]:
rows = {'always "real" (majority)': {'accuracy %': round(100 * y.mean(), 1), 'balanced accuracy %': 50.0,
                                     'AUC': 0.5, 'predicted real %': 100.0}}
for name, p in P.items():
    m = evaluate(y, p.to_numpy())
    rows[name] = {k: m[k] for k in ('accuracy %', 'balanced accuracy %', 'AUC', 'predicted real %')}
    rows[name]['AUC 95% CI'] = bootstrap_ci(y, p.to_numpy())['AUC 95% CI']
pd.DataFrame(rows).T

,accuracy %,balanced accuracy %,AUC,predicted real %,AUC 95% CI
"always ""real"" (majority)",56.1,50.0,0.5,100.0,NaN
"LSTM, my paper (A0, 3 seeds mean)",55.6,58.5,0.646,27.2,"(0.618, 0.674)"
"BiLSTM tuned (A6, 3 seeds mean)",56.4,59.3,0.684,27.1,"(0.656, 0.71)"
TF-IDF (ISOT),57.5,61.2,0.718,20.9,"(0.692, 0.744)"
"DistilBERT (ISOT, 3 seeds mean)",59.6,62.7,0.707,26.0,"(0.68, 0.732)"
"Haiku 4.5, zero-shot",77.5,75.2,0.878,71.7,"(0.861, 0.895)"
"Sonnet 5.5, zero-shot",78.3,75.6,0.929,75.1,"(0.917, 0.94)"


In [3]:
def diff(a, b):
    r = paired_bootstrap_auc_diff(y, P[a].to_numpy(), P[b].to_numpy())
    return {'A': a, 'B': b, 'AUC(A) - AUC(B)': r['AUC difference (A - B)'], '95% CI': r['95% CI']}

pd.DataFrame([diff('Sonnet 5.5, zero-shot', 'DistilBERT (ISOT, 3 seeds mean)'),
              diff('Sonnet 5.5, zero-shot', 'TF-IDF (ISOT)'),
              diff('Haiku 4.5, zero-shot', 'TF-IDF (ISOT)'),
              diff('Sonnet 5.5, zero-shot', 'Haiku 4.5, zero-shot'),
              diff('Sonnet 5.5, zero-shot', 'LSTM, my paper (A0, 3 seeds mean)')])

,A,B,AUC(A) - AUC(B),95% CI
0,"Sonnet 5.5, zero-shot","DistilBERT (ISOT, 3 seeds mean)",0.222,"(0.196, 0.248)"
1,"Sonnet 5.5, zero-shot",TF-IDF (ISOT),0.211,"(0.185, 0.235)"
2,"Haiku 4.5, zero-shot",TF-IDF (ISOT),0.161,"(0.136, 0.185)"
3,"Sonnet 5.5, zero-shot","Haiku 4.5, zero-shot",0.050,"(0.037, 0.063)"
4,"Sonnet 5.5, zero-shot","LSTM, my paper (A0, 3 seeds mean)",0.282,"(0.254, 0.312)"


## 3. Which articles are accepted as real? / 哪些文章被判为真

ISOT 模型最典型的失败是把纽约时报的真新闻判为假（`05`）。按出版方分组，看每个模型把多少文章判为"真"。真新闻组越高越好，假新闻组越低越好。

In [4]:
order = ['REAL: The New York Times', 'REAL: Breitbart', 'REAL: other outlets', 'FAKE articles (all)']
table = pd.DataFrame({name: (p > 0.5).groupby(group).mean().mul(100).round(1).reindex(order) for name, p in P.items()}).T
table.columns = [f'{c} (n={int((group == c).sum())})' for c in order]
table

,REAL: The New York Times (n=400),REAL: Breitbart (n=141),REAL: other outlets (n=301),FAKE articles (all) (n=659)
"LSTM, my paper (A0, 3 seeds mean)",39.2,27.7,31.9,17.8
"BiLSTM tuned (A6, 3 seeds mean)",44.5,18.4,30.9,16.7
TF-IDF (ISOT),43.2,14.9,21.6,8.3
"DistilBERT (ISOT, 3 seeds mean)",50.2,18.4,28.6,11.7
"Haiku 4.5, zero-shot",99.2,82.3,92.0,43.4
"Sonnet 5.5, zero-shot",99.8,95.0,96.0,46.4


## 4. Calibration-free view: AUC within each real outlet / 不依赖阈值：每个真新闻来源对假新闻的 AUC

判为真的比例取决于 0.5 这个阈值；大模型没有在 WELFake 上校准过。AUC 只看排序：随机抽一篇这个来源的真新闻和一篇假新闻，模型给真新闻打分更高的概率。

In [5]:
fake = (group == 'FAKE articles (all)').to_numpy()
rows = {}
for outlet in order[:3]:
    m = ((group == outlet).to_numpy()) | fake
    rows[outlet] = {name: round(roc_auc_score(y[m], p[m]), 3) for name, p in P.items()}
pd.DataFrame(rows)

,REAL: The New York Times,REAL: Breitbart,REAL: other outlets
"LSTM, my paper (A0, 3 seeds mean)",0.672,0.602,0.634
"BiLSTM tuned (A6, 3 seeds mean)",0.748,0.593,0.643
TF-IDF (ISOT),0.792,0.615,0.666
"DistilBERT (ISOT, 3 seeds mean)",0.788,0.599,0.650
"Haiku 4.5, zero-shot",0.946,0.751,0.848
"Sonnet 5.5, zero-shot",0.983,0.836,0.900


## 5. What the numbers mean / 结果说明

**1. 大模型基本避开了 ISOT 的陷阱。** ISOT 模型把一半以上的纽约时报真新闻判为假（只接受 39–50%），因为它们"不像路透社"；Haiku 和 Sonnet 接受了 99% 以上。AUC 从 0.65–0.72 提高到 0.88（Haiku）和 0.93（Sonnet），Sonnet 比 DistilBERT 可靠地高 0.22，比你论文中的 LSTM 高 0.28。原因很直接：大模型 **没有在按来源打标签的数据上训练过**，所以没有学到"认来源"这条捷径。

**2. 知识更多的模型更好**：Sonnet 比 Haiku 可靠地高 0.050，和 LIAR 上的结论一致。

**3. 阈值偏宽松**：两个大模型都把约 45% 的假新闻判为真，所以准确率只有约 78%，而 AUC 很高。也就是说，它们的 *排序* 很好，但分界线放得太低；如果在 WELFake val 上校准阈值，准确率会明显提高。这里没有做，因为那需要在 val 上再运行一次。

**4. 不是简单地排斥党派媒体**：Breitbart 在 WELFake 中被标为"真"，但立场鲜明，ISOT 模型只接受 15–28%；Sonnet 接受了 95%。不过 Breitbart 和假新闻之间的 AUC（0.836）明显低于纽约时报和假新闻之间（0.983）：党派媒体和假新闻网站在话题和措辞上更接近，更难区分。

**5. 和第 4 步比较**：第 4 步把 WELFake 的其他来源加入训练（多来源训练），在"留出纽约时报"的测试上达到 AUC 0.968–0.975。这里 Sonnet **不用任何训练数据**，纽约时报对假新闻的 AUC 是 0.983。两者的测试文章不完全相同，只能粗略比较，但说明零样本的大模型至少和多来源训练一样能避开来源捷径。

1. The LLMs largely avoid the ISOT trap: they accept over 99% of NYT articles (ISOT models 39–50%), and AUC rises from 0.65–0.72 to 0.88 (Haiku) and 0.93 (Sonnet), because they were never trained on source-labelled data. 2. Sonnet beats Haiku by 0.05. 3. Both accept ~45% of fakes: good ranking, threshold too low. 4. They do not simply reject partisan outlets (Sonnet accepts 95% of Breitbart), but Breitbart vs fake is harder (AUC 0.836) than NYT vs fake (0.983). 5. Without any training data, Sonnet's NYT-vs-fake AUC (0.983) is roughly on par with step 4's multi-source training (0.968–0.975 on a different NYT test set).

## 6. Conclusions and caveats / 结论与注意事项

**结论：在最初的跨来源问题上，零样本的大模型远好于在 ISOT 上训练的所有模型。** 在同样的 1,501 篇 WELFake 文章上，你论文中的 LSTM 是 AUC 0.646，DistilBERT 0.707，Sonnet 0.929；纽约时报的真新闻从约一半被判为假，变成几乎全部判为真。根本原因是：大模型没有在"标签 = 来源"的数据上训练，所以没有学到那条捷径。

**但这个结果需要谨慎解读，有三个无法排除的因素：**

1. **记忆和知识无法分开。** WELFake 的文章来自 2016–2017 年，几乎可以肯定在大模型的训练数据里（尤其是纽约时报）。我们尝试了文本污染检测的常用方法（让模型逐字续写文章），但在小样本测试中 Sonnet 每次都拒绝复现已发表的原文，所以这个方法无法使用，没有全量运行。"知道某件真实的事发生过"本来就是核查事实需要的知识，但"认得这篇文章"不是。**真正的检验是在模型训练之后发表的新闻上测试**，WELFake 做不到这一点。
2. **WELFake 的标签也是按网站打的。** 大模型可能部分是在判断"文风像不像假新闻网站"。它接受 95% 的 Breitbart 文章，说明它没有简单地排斥党派媒体，但不能完全排除文风的作用。
3. **其他**：只抽样了 1,501 篇（test 共 4,691 篇）；只用了标题和前 300 个词；Sonnet 没有 temperature 0；阈值没有校准（准确率约 78%，AUC 0.93）。

**对项目的意义**：原论文的 99% 来自"认路透社"；在 ISOT 上训练的任何模型（LSTM、BiLSTM、DistilBERT）都无法摆脱这个问题，换数据（第 4 步的多来源训练）或者不在这种数据上训练（这里的零样本大模型）才能避开它。不过大模型在 2016 年新闻上的高分，有多少能延续到它没见过的新闻上，仍然是一个开放的问题。

---

**Zero-shot LLMs are far better than every ISOT-trained model on the original cross-source problem** (Sonnet AUC 0.929 vs DistilBERT 0.707 and the paper's LSTM 0.646 on the same articles; NYT acceptance from ~half to ~100%), because they never learned the source shortcut. Caveats: memory cannot be separated from knowledge for these 2016–17 articles (the continuation test failed because the model declines to reproduce published text), WELFake labels are per site, and only a 1,501-article sample with untuned thresholds was used. Whether the gain carries over to news published after the models' training remains open.